In [1]:
import tensorflow as tf
import tensorflow_recommenders as tfrs

import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import os
import array
import collections

from typing import Dict, List, Optional, Text, Tuple

In [2]:
port_size = 20

In [3]:
retriever_location_ = r"D:\dev work\recommender systems\Atrad_CARS\model_weights\2024_06_24_04\retriever_v3_port_v2__fixed_port_size_20"
ranking_location_ = r"D:\dev work\recommender systems\Atrad_CARS\model_weights\2024_05_27\tf_listwise_ranking_2024_05_27_11_20"
stock_info_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\stock_data.xlsx"

test_ds_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_port_size_{}\retriver_test".format(port_size)
train_ds_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_port_size_{}\retriver_train".format(port_size)

portfolios_loc = r"D:/dev work/recommender systems/Atrad_CARS/data/portfolios_v2_fixed_port_size_{}/portfolios".format(port_size)

results_loc = r"D:\dev work\recommender systems\Atrad_CARS\results"

In [4]:
test_ds = tf.data.Dataset.load(test_ds_loc).cache()

train_ds = tf.data.Dataset.load(train_ds_loc).cache()

portfolios = tf.data.Dataset.load(portfolios_loc).cache()

In [5]:
from retrieval_recommender_v2 import Retriever

retriever = Retriever(
    use_timestamp = True,
    portfolios = portfolios
)

retriever.load_weights(retriever_location_)

retriever.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))


In [6]:
from ranker_recommender import Ranker

ranker = Ranker(
    loss = tf.keras.losses.MeanSquaredError(),
    portfolios = portfolios
)

ranker.load_weights(ranking_location_)
ranker.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))


In [7]:
stock_info = pd.read_excel(stock_info_loc)
stock_info = stock_info.drop(['Unnamed: 0','buisnesssummary'],axis = 1)
stock_info = stock_info.rename(columns = {
    'symbol':'STOCKCODE',
    'name' : 'STOCKNAME',
    'gics_code' : 'GICS'
})
stock_info = stock_info[~stock_info['GICS'].isna()]

stock_info.shape
print("items data shape :: {}".format(stock_info.shape))
unique_items_ = np.unique(np.concatenate(list(train_ds.batch(1000).map(lambda x: x["STOCKCODE"]).as_numpy_iterator())))
stock_info = stock_info[stock_info['STOCKCODE'].isin([item.decode('utf-8') for item in unique_items_])]

items_ds = tf.data.Dataset.from_tensor_slices(stock_info.to_dict(orient= 'list'))

items data shape :: (280, 3)


# evaluation function

In [8]:
def evaluate(retriever,
             test: tf.data.Dataset,
             train: Optional[tf.data.Dataset] = None,
             timestamp: int = datetime.timestamp(datetime.now()),
             k: int = 10):
  
  item_ids = np.concatenate(list(items_ds.batch(1000).map(lambda x: x["STOCKCODE"]).as_numpy_iterator()))

  item_vocabulary = dict(zip(item_ids.tolist(), range(len(item_ids))))
  item_vocabulary_inv = {v: k for k, v in item_vocabulary.items()}

  train_user_to_items = collections.defaultdict(lambda: array.array("i"))
  test_user_to_items = collections.defaultdict(lambda: array.array("i"))

  if train is not None:
    for row in train.as_numpy_iterator():
      user_id = row["CDSACCNO"]
      item_id = item_vocabulary[row["STOCKCODE"]]
      train_user_to_items[user_id].append(item_id)

  for row in test.as_numpy_iterator():
    user_id = row["CDSACCNO"]
    item_id = item_vocabulary[row["STOCKCODE"]]
    test_user_to_items[user_id].append(item_id)

  item_embeddings = np.concatenate(list(items_ds.batch(len(items_ds)).map(lambda x: retriever.item_model(x)).as_numpy_iterator()))

  user_ids = []
  precision_values = []
  recall_values = []
  num_test_items = []
  num_train_items = []
  recommendations = []

  for user_id, test_items in tqdm(test_user_to_items.items()):
    user_embedding = retriever.user_model(
      {
        'CDSACCNO' : tf.constant([user_id]),
        'UNIX_TS' : tf.constant([timestamp])
      }
      ).numpy()
    scores = (user_embedding @ item_embeddings.T).flatten()

    test_items = np.frombuffer(test_items, dtype=np.int32)
    
    if train is not None:
      train_items = np.frombuffer(
          train_user_to_items[user_id], dtype=np.int32)
      scores[train_items] = -1e6

    

    top_items = np.argsort(-scores)[:k]
    recommendations.append([item_vocabulary_inv[item_id].decode('utf-8') for item_id in top_items])

    num_test_items_in_k = sum(x in top_items for x in  test_items)
    precision_values.append(num_test_items_in_k / k)
    
    recall_values.append(num_test_items_in_k / len(test_items))
    num_test_items.append(len((test_items)))
    num_train_items.append(len(train_user_to_items[user_id]))
    user_ids.append(user_id)

  results_df_ = pd.DataFrame(
    columns = ['CDSACCNO','precision@k', 'recall@k','num_test_items','portfolio_size', 'recommendations'],
    data = list(zip(user_ids, precision_values, recall_values, num_test_items, num_train_items, recommendations))
  )

  return {
      "precision_at_k": np.mean(precision_values),
      "recall_at_k": np.mean(recall_values),
      "results_df_" : results_df_
  }

In [9]:
results = evaluate(
    retriever,
    test_ds,
    train_ds
)

100%|██████████| 2796/2796 [00:19<00:00, 145.33it/s]


In [10]:
results['precision_at_k'] , results['recall_at_k']

(0.02861230329041488, 0.0715307582260372)

In [11]:
results['results_df_']['CDSACCNO'] = results['results_df_']['CDSACCNO'].apply(lambda x: x.decode('utf-8'))
results['results_df_']

,CDSACCNO,precision@k,recall@k,num_test_items,portfolio_size,recommendations
0,BMS-20157-LI/00,0.0,0.0,4,16,"[UAL, DIAL, ALLI, HASU, PLR, ASCO, DPL, REG, K..."
1,HDF-863433169-VN/00,0.0,0.0,4,16,"[REG, CALT, DPL, MDL, DIAL, UAL, RICH, CIND, R..."
2,BMS-26083-LI/00,0.0,0.0,4,16,"[MDL, SINI, CTEA, REG, TESS, DIMO, SLTL, VLL, ..."
3,HDF-44052-LI/00,0.0,0.0,4,16,"[DIAL, PINS, CALF, CARG, CINS, KFP, TSML, COMB..."
4,BMS-911262550-VN/00,0.0,0.0,4,16,"[CSF, ACAP, MDL, RGEM, RCH, RICH, CALT, DPL, H..."
...,...,...,...,...,...,...
2791,BMS-833490346-VN/00,0.0,0.0,4,16,"[PINS, CINS, DIAL, CSD, HASU, SINI, REG, GREG,..."
2792,BMS-853001554-VN/00,0.0,0.0,4,16,"[MDL, DIAL, PLR, CIND, AMF, SINI, ACAP, PINS, ..."
2793,BMS-85992-LI/00,0.0,0.0,4,16,"[DIAL, TRAN, KDL, RGEM, KHC, JINS, RCH, BLUE, ..."
2794,BMS-860530309-VN/00,0.0,0.0,4,16,"[KDL, NEH, UAL, MDL, HUNA, DIAL, CLND, PLR, MH..."


In [12]:
results_loc_ = r"D:\dev work\recommender systems\Atrad_CARS\results"

retriever_name = os.path.basename(retriever_location_)
ranker_name = os.path.basename(ranking_location_)

results_file_name = retriever_name + "_&_" + ranker_name + "_results_fixed_port_{}.csv".format(port_size)

results_save_path = os.path.join(results_loc_, results_file_name)
# results_save_path
results['results_df_'].to_csv(results_save_path, index = False)